In [9]:
import duckdb
import pandas as pd
import numpy as np

# ==========================================
# 0. ADIM: JUPYTER / PANDAS GÖRÜNÜM AYARLARI
# ==========================================
# Tabloların kırılmasını önler, tüm sütunları gösterir ve net sayı formatına çevirir.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


# ==========================================
# 1. ADIM: DUCKDB İLE SIKILAŞTIRILMIŞ SÜZME
# ==========================================
# paysim_dataset.csv dosyasından tam bakiye boşaltma (Drain) kurallarıyla süzme yapılır.

sql_query = """
SELECT 
    step,
    type,
    amount,
    nameOrig AS sender_id,
    oldbalanceOrg AS balance_before,
    newbalanceOrig AS balance_after,
    nameDest AS receiver_id,
    oldbalanceDest AS dest_balance_before,
    newbalanceDest AS dest_balance_after,
    isFraud
FROM read_csv_auto('paysim dataset.csv')
WHERE type IN ('CASH_OUT', 'TRANSFER')
  -- 1. Gönderici Hesabının BİREBİR TAMAMEN Boşaltılması
  AND amount = oldbalanceOrg 
  
  -- 2. İşlem sonrası gönderici bakiyesinin sıfırlanması
  AND newbalanceOrig = 0
  
  -- 3. Sıfır bakiyeli hesapları filtreleme
  AND oldbalanceOrg > 0
  
  -- 4. Alıcı Tarafı PaySim Anomali İzi (Gelen bakiye güncellenmiyor)
  AND oldbalanceDest = 0 
  AND newbalanceDest = 0
  
  -- 5. Yüksek Tutar Barajı (Yönetilebilir vaka sayısı için)
  AND amount >= 1000000
ORDER BY amount DESC;
"""

# Sorguyu çalıştırıp Pandas DataFrame'e dönüştürme
drain_df = duckdb.query(sql_query).to_df()

print(f"✅ DuckDB İle Süzülen Yönetilebilir Şüpheli İşlem Sayısı: {len(drain_df)}\n")
display(drain_df.head())


# ==========================================
# 2. ADIM: PANDAS İLE İNCELEME VE PERFORMANS RAPORU
# ==========================================

# A. Gerçek Fraud (isFraud == 1) Oranını İnceleme
fraud_breakdown = drain_df.groupby('type')['isFraud'].value_counts(normalize=True).unstack(fill_value=0) * 100
print("\n--- İşlem Tipine Göre Gerçek Fraud Dağılımı (%) ---")
display(fraud_breakdown.round(2))

# B. Boşaltılan Toplam ve Precision (Hassasiyet) İstatistikleri
stats_summary = drain_df.groupby('type').agg(
    total_drained_amount=('amount', 'sum'),
    avg_drained_amount=('amount', 'mean'),
    max_drained_amount=('amount', 'max'),
    total_alerts=('sender_id', 'count'),
    total_actual_frauds=('isFraud', 'sum')
).reset_index()

# Gerçek Fraud Yakalama Başarısı (Precision %)
stats_summary['fraud_precision_pct'] = (stats_summary['total_actual_frauds'] / stats_summary['total_alerts']) * 100

print("\n--- Rapid Balance Drain Özeti ---")
display(stats_summary)

# C. En Yüksek Hacimli Boşaltılan İlk 10 Hesabı Çıkarma
top_drain_cases = drain_df.sort_values(by='amount', ascending=False).head(10)
print("\n--- En Yüksek Hacimli Boşaltılan İlk 10 Hesap ---")
display(top_drain_cases)


# ==========================================
# 3. ADIM: EXCEL / RAPOR AKTARIMI (İSTEĞE BAĞLI)
# ==========================================
# drain_df.to_excel("supheli_islem_listesi.xlsx", index=False)

✅ DuckDB İle Süzülen Yönetilebilir Şüpheli İşlem Sayısı: 1198



,step,type,amount,sender_id,balance_before,balance_after,receiver_id,dest_balance_before,dest_balance_after,isFraud
0,651,TRANSFER,9996886.64,C429282035,9996886.64,0.00,C1318460385,0.00,0.00,1
1,46,TRANSFER,9977761.05,C154757729,9977761.05,0.00,C1895716929,0.00,0.00,1
2,226,TRANSFER,9960382.40,C427148786,9960382.40,0.00,C1532977706,0.00,0.00,1
3,33,TRANSFER,9887819.06,C1548903046,9887819.06,0.00,C2065423383,0.00,0.00,1
4,256,TRANSFER,9811104.49,C1983750370,9811104.49,0.00,C381637925,0.00,0.00,1



--- İşlem Tipine Göre Gerçek Fraud Dağılımı (%) ---


isFraud,1
type,
TRANSFER,100.00



--- Rapid Balance Drain Özeti ---


,type,total_drained_amount,avg_drained_amount,max_drained_amount,total_alerts,total_actual_frauds,fraud_precision_pct
0,TRANSFER,3761182376.28,3139551.23,9996886.64,1198,1198,100.00



--- En Yüksek Hacimli Boşaltılan İlk 10 Hesap ---


,step,type,amount,sender_id,balance_before,balance_after,receiver_id,dest_balance_before,dest_balance_after,isFraud
0,651,TRANSFER,9996886.64,C429282035,9996886.64,0.00,C1318460385,0.00,0.00,1
1,46,TRANSFER,9977761.05,C154757729,9977761.05,0.00,C1895716929,0.00,0.00,1
2,226,TRANSFER,9960382.40,C427148786,9960382.40,0.00,C1532977706,0.00,0.00,1
3,33,TRANSFER,9887819.06,C1548903046,9887819.06,0.00,C2065423383,0.00,0.00,1
4,256,TRANSFER,9811104.49,C1983750370,9811104.49,0.00,C381637925,0.00,0.00,1
5,351,TRANSFER,9772559.35,C417981063,9772559.35,0.00,C1176673721,0.00,0.00,1
6,718,TRANSFER,9749042.95,C1040320190,9749042.95,0.00,C598263954,0.00,0.00,1
7,398,TRANSFER,9725837.08,C2078773097,9725837.08,0.00,C775810552,0.00,0.00,1
8,477,TRANSFER,9639524.70,C1490771263,9639524.70,0.00,C486822956,0.00,0.00,1
9,124,TRANSFER,9639052.83,C167488694,9639052.83,0.00,C629818525,0.00,0.00,1
